# Trajectory Metrics From NPZ Archives

Recompute the teacher, all-layer cotrained EoS, and corrected CM comparison directly from saved trajectories. No model loading, GPU, or evaluation JSON is required.

All four metrics use **XY coordinates in metres**, over all 64 saved timesteps, with equal weight per clip:

- **ADE**: time-averaged Euclidean error of draw 0, then mean across clips. This is **not** the mean ADE of all six draws.
- **min_ADE**: smallest trajectory ADE among the six draws for each clip, then mean across clips.
- **max_ADE**: largest trajectory ADE among the six draws for each clip, then mean across clips. This is **not** the maximum instantaneous error or the worst clip.
- **center**: ADE of the trajectory obtained by averaging the six predicted XY trajectories, then mean across clips.

The loader checks 1,000 unique matching clip IDs, exactly matching saved 3D ground truth, six draws, and finite values. These archives contain complete fixed-horizon trajectories; no validity masking is applied. Metre units and ego-frame coordinates are assumptions of the evaluation exporter, not inferred from the numeric arrays.

**Comparability:** matching clips and ground truth does not guarantee identical inputs. Navigation-distance stripping was not verified for some historical runs; provenance is shown below. All EoS and CM rows use epoch 2 (`checkpoint-6876`). CM rows use the corrected fp16-training/fp32-master EMA checkpoints; only NFE=1 has been measured for these CM models.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

TRAINING_ROOT = Path(
    "/temp/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training"
)
EXPECTED_CLIPS = 1000
EXPECTED_SAMPLES = 6
EXPECTED_TIMESTEPS = 64
METRIC_COLUMNS = ["ADE", "min_ADE", "max_ADE", "center"]


In [ ]:

runs = [
    ("Alpamayo 1.5", 1, "teacher15_nav_fp32archive_h100c_eos_models--nvidia--Alpamayo-1.5-10B-A1-format_nfe1", True),
    ("Alpamayo 1.5", 2, "teacher15_nav_fp32archive_h100c_eos_models--nvidia--Alpamayo-1.5-10B-A1-format_nfe2", True),
    ("Alpamayo 1.5", 6, "teacher15_2cam_nav_stripped_eos_models--nvidia--Alpamayo-1.5-10B-A1-format_nfe6", True),
    ("Alpamayo 1.5", 10, "teacher_eval/teachernav_k10", False),
    ("KD 4B", 1, "kd4b_navstripped_h100c_control_checkpoint-13752_nfe1", True),
    ("KD 4B", 2, "kd4b_navstripped_h100c_control_checkpoint-13752_nfe2", True),
    ("KD 4B", 10, "stitch_4b_nav4bspan2camallfc_m1-9-18-36_framecache1080p_checkpoint-13752_cam13_nav", False),
    ("KD 2B", 1, "kd2b_navstripped_h100c_control_checkpoint-13752_nfe1", True),
    ("KD 2B", 2, "kd2b_navstripped_h100c_control_checkpoint-13752_nfe2", True),
    ("KD 2B", 10, "stitch_2b_mixspan2bnavfc_m1-9-18-36_framecache1080p_checkpoint-13752_cam13_nav", False),
    ("EoS 4B", 1, "cotrain4b_all36_eos_checkpoint-6876_nfe1", False),
    ("EoS 4B", 2, "eos4b_cm_matched_reference_eos_checkpoint-6876_nfe2", True),
    ("EoS 4B", 6, "cotrain4b_all36_eos_checkpoint-6876_nfe6", True),
    ("EoS 4B", 10, "cotrain4b_all36_eos_checkpoint-6876", False),
    ("EoS 2B", 1, "cotrain2b_all28_eos_checkpoint-6876_nfe1", False),
    ("EoS 2B", 2, "cotrain2b_all28_eos_checkpoint-6876_nfe2", False),
    ("EoS 2B", 6, "cotrain2b_all28_navstripped_h100c_eos_checkpoint-6876_nfe6", True),
    ("EoS 2B", 10, "cotrain2b_all28_navstripped_h100c_eos_checkpoint-6876", True),
    ("CM 4B", 1, "cm4b_fp16_master32_eos_checkpoint-6876_nfe1", True),
    ("CM 2B", 1, "cm2b_all28_fp16_master32_h100c_eos_checkpoint-6876_nfe1", True),
]

# runs = [run for run in runs if run[1] <= 1]  # Filter to only NFE <= 1 

manifest = pd.DataFrame(runs, columns=["Model", "NFE", "Archive stem", "Nav stripping verified"])
manifest["NPZ"] = manifest["Archive stem"].map(lambda stem: TRAINING_ROOT / f"{stem}.npz")
manifest["Navigation distance stripping"] = manifest["Nav stripping verified"].map(
    {True: "Explicitly enabled", False: "Historical: not verified"}
)
if manifest.duplicated(["Model", "NFE"]).any():
    raise ValueError("Duplicate model/NFE rows in the archive manifest")
missing = [str(path) for path in manifest["NPZ"] if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing NPZ archives; update TRAINING_ROOT or runs:\n" + "\n".join(missing))
print(f"Found all {len(manifest)} NPZ archives.")

In [2]:
def load_archive(path, expected_nfe):
    """Load one fixed-format evaluation archive and sort its rows by clip ID."""
    path = Path(path)
    with np.load(path, allow_pickle=False) as archive:
        required = {"pred_xyz", "gt_xyz"}
        if not required.issubset(archive.files):
            raise ValueError(f"{path}: missing arrays {sorted(required - set(archive.files))}")
        id_key = next((key for key in ("clip_ids", "clip_id") if key in archive), None)
        if id_key is None:
            raise ValueError(f"{path}: missing clip_ids or clip_id")
        clip_ids = archive[id_key].astype(str)
        predictions = archive["pred_xyz"]
        ground_truth = archive["gt_xyz"]
        expected_pred = (EXPECTED_CLIPS, 1, EXPECTED_SAMPLES, EXPECTED_TIMESTEPS, 3)
        expected_gt = (EXPECTED_CLIPS, EXPECTED_TIMESTEPS, 3)
        if clip_ids.shape != (EXPECTED_CLIPS,) or len(np.unique(clip_ids)) != EXPECTED_CLIPS:
            raise ValueError(f"{path}: expected {EXPECTED_CLIPS} unique clip IDs")
        if predictions.shape != expected_pred or ground_truth.shape != expected_gt:
            raise ValueError(
                f"{path}: got pred {predictions.shape}, GT {ground_truth.shape}; "
                f"expected {expected_pred}, {expected_gt}"
            )
        if not np.isfinite(predictions).all() or not np.isfinite(ground_truth).all():
            raise ValueError(f"{path}: non-finite predictions or ground truth")
        if "cameras" in archive and not np.array_equal(archive["cameras"], [1, 3]):
            raise ValueError(f"{path}: camera metadata does not match [1, 3]")
        if "inference_step" in archive:
            step = archive["inference_step"]
            if step.size != 1 or step.item() != expected_nfe:
                raise ValueError(f"{path}: NFE metadata does not match {expected_nfe}")
        order = np.argsort(clip_ids)
        inspection = {
            "Keys": ", ".join(archive.files),
            "Prediction shape": str(predictions.shape),
            "Prediction dtype": str(predictions.dtype),
            "GT shape": str(ground_truth.shape),
            "GT dtype": str(ground_truth.dtype),
            "NFE metadata": "verified" if "inference_step" in archive else "absent; manifest/log provenance",
            "Camera metadata": "verified" if "cameras" in archive else "absent; manifest/log provenance",
        }
        return (
            clip_ids[order],
            predictions[order, 0, ..., :2].astype(np.float64),
            ground_truth[order],
            inspection,
        )


def compute_metrics(pred_xy, gt_xyz):
    """Per-clip full-horizon XY metrics; reduce time before choosing min/max draw."""
    pred_xy = np.asarray(pred_xy, dtype=np.float64)
    gt_xyz = np.asarray(gt_xyz, dtype=np.float64)
    if (
        pred_xy.ndim != 4 or gt_xyz.ndim != 3
        or pred_xy.shape[-1] != 2 or gt_xyz.shape[-1] not in (2, 3)
        or pred_xy.shape[0] != gt_xyz.shape[0]
        or pred_xy.shape[2] != gt_xyz.shape[1]
        or any(size == 0 for size in pred_xy.shape)
    ):
        raise ValueError("Expected predictions [clips, draws, time, 2] and aligned GT [clips, time, 2 or 3]")
    if not np.isfinite(pred_xy).all() or not np.isfinite(gt_xyz).all():
        raise ValueError("Metrics require finite predictions and ground truth")
    target_xy = gt_xyz[..., :2]
    per_draw_ade = np.linalg.norm(pred_xy - target_xy[:, None], axis=-1).mean(axis=-1)
    mean_trajectory = pred_xy.mean(axis=1)
    return pd.DataFrame({
        "ADE": per_draw_ade[:, 0],
        "min_ADE": per_draw_ade.min(axis=1),
        "max_ADE": per_draw_ade.max(axis=1),
        "center": np.linalg.norm(mean_trajectory - target_xy, axis=-1).mean(axis=-1),
    })


def check_metric_definitions():
    truth = np.zeros((2, 2, 3))
    prediction = np.zeros((2, 2, 2, 2))
    prediction[0, 0, :, 0] = [0, 4]
    prediction[0, 1, :, 0] = [3, 0]
    prediction[1, 0, :, 0] = [1, 1]
    prediction[1, 1, :, 0] = [-1, -1]
    measured = compute_metrics(prediction, truth)
    np.testing.assert_allclose(measured["ADE"], [2, 1])
    np.testing.assert_allclose(measured["min_ADE"], [1.5, 1])
    np.testing.assert_allclose(measured["max_ADE"], [2, 1])
    np.testing.assert_allclose(measured["center"], [1.75, 0])
    np.testing.assert_allclose(measured[METRIC_COLUMNS].mean(), [1.5, 1.25, 1.5, 0.875])
    for bad_prediction in (prediction[:, :, :1], prediction * np.nan):
        try:
            compute_metrics(bad_prediction, truth)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid input was not rejected")
    print("Synthetic metric checks passed.")


check_metric_definitions()

Synthetic metric checks passed.


In [3]:
summary_rows = []
inspection_rows = []
per_clip_metrics = {}
reference_ids = None
reference_gt = None
reference_path = None

for record in manifest.to_dict("records"):
    path = record["NPZ"]
    clip_ids, pred_xy, gt_xyz, inspection = load_archive(path, record["NFE"])
    if reference_ids is None:
        reference_ids = clip_ids.copy()
        reference_gt = gt_xyz.copy()
        reference_path = path
    else:
        if not np.array_equal(clip_ids, reference_ids):
            raise ValueError(f"{path}: clip IDs differ from {reference_path}")
        if not np.array_equal(gt_xyz, reference_gt):
            raise ValueError(f"{path}: ground truth differs from {reference_path}")
    metrics = compute_metrics(pred_xy, gt_xyz)
    averages = metrics[METRIC_COLUMNS].mean()
    metrics.insert(0, "clip_id", clip_ids)
    per_clip_metrics[(record["Model"], record["NFE"])] = metrics
    summary_rows.append({
        "Model": record["Model"],
        "NFE": record["NFE"],
        **averages.to_dict(),
    })
    inspection_rows.append({
        "Model": record["Model"], "NFE": record["NFE"],
        "Clips": len(clip_ids), **inspection,
    })

results = pd.DataFrame(summary_rows, columns=["Model", "NFE", *METRIC_COLUMNS])
archive_inspection = pd.DataFrame(inspection_rows)
print(
    f"Verified {len(results)} archives: {len(reference_ids):,} matching clips, "
    f"identical saved GT, {EXPECTED_SAMPLES} draws and {EXPECTED_TIMESTEPS} timesteps per clip."
)
comparison_table = (
    results.style
    .format({metric: "{:.2f}" for metric in METRIC_COLUMNS})
    .hide(axis="index")
    .set_caption("NPZ-derived trajectory metrics: XY metres; green = lower, red = higher; bold = column minimum")
    .set_properties(subset=METRIC_COLUMNS, **{"text-align": "right"})
    .background_gradient(cmap="RdYlGn_r", subset=METRIC_COLUMNS, axis=0)
    .highlight_min(subset=METRIC_COLUMNS, axis=0, props="font-weight: bold;")
)
display(comparison_table)

Verified 20 archives: 1,000 matching clips, identical saved GT, 6 draws and 64 timesteps per clip.


Model,NFE,ADE,min_ADE,max_ADE,center
Alpamayo 1.5,1,1.71,1.31,2.21,1.62
Alpamayo 1.5,2,1.47,0.95,2.14,1.36
Alpamayo 1.5,6,1.61,0.75,2.83,1.35
Alpamayo 1.5,10,1.70,0.74,3.30,1.39
KD 4B,1,2.13,1.76,2.49,2.09
KD 4B,2,1.77,1.17,2.45,1.64
KD 4B,10,2.04,0.90,3.71,1.65
KD 2B,1,2.15,1.84,2.49,2.11
KD 2B,2,1.99,1.33,2.78,1.87
KD 2B,10,2.37,1.03,4.28,1.93


In [6]:
SELECTED_MODELS = ["Alpamayo 1.5", "EoS 4B", "KD 4B", "CM 4B"]
SELECTED_NFE = [1]

filtered_results = results.loc[
    results["Model"].isin(SELECTED_MODELS) & results["NFE"].isin(SELECTED_NFE)
].copy()

filtered_table = (
    filtered_results.style
    .use(comparison_table.export())
    .format({metric: "{:.2f}" for metric in METRIC_COLUMNS})
    .hide(axis="index")
    .set_caption("Filtered trajectory metrics: XY metres; green = lower, red = higher; bold = column minimum")
)
display(filtered_table)

Model,NFE,ADE,min_ADE,max_ADE,center
Alpamayo 1.5,1,1.71,1.31,2.21,1.62
KD 4B,1,2.13,1.76,2.49,2.09
EoS 4B,1,1.59,1.34,1.86,1.54
CM 4B,1,1.54,1.04,2.07,1.45


In [4]:
provenance = manifest[["Model", "NFE", "NPZ", "Navigation distance stripping"]].copy()
provenance["NPZ"] = provenance["NPZ"].map(str)
display(provenance.style.hide(axis="index").set_caption("Archive provenance and preprocessing caveats"))
display(archive_inspection.style.hide(axis="index").set_caption("Loaded array structure and metadata checks"))

EXPORT_CSV = False
CSV_PATH = TRAINING_ROOT / "npz_metrics_comparison_xy.csv"
if EXPORT_CSV:
    export_table = results.merge(provenance, on=["Model", "NFE"], validate="one_to_one")
    export_table.to_csv(CSV_PATH, index=False, float_format="%.8f")
    print(f"Exported {CSV_PATH}")

Model,NFE,NPZ,Navigation distance stripping
Alpamayo 1.5,1,/temp/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training/teacher15_nav_fp32archive_h100c_eos_models--nvidia--Alpamayo-1.5-10B-A1-format_nfe1.npz,Explicitly enabled
EoS 4B,1,/temp/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training/cotrain4b_all36_eos_checkpoint-6876_nfe1.npz,Historical: not verified
EoS 2B,1,/temp/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training/cotrain2b_all28_eos_checkpoint-6876_nfe1.npz,Historical: not verified
CM 4B,1,/temp/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training/cm4b_fp16_master32_eos_checkpoint-6876_nfe1.npz,Explicitly enabled
CM 2B,1,/temp/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/training/cm2b_all28_fp16_master32_h100c_eos_checkpoint-6876_nfe1.npz,Explicitly enabled


Model,NFE,Clips,Keys,Prediction shape,Prediction dtype,GT shape,GT dtype,NFE metadata,Camera metadata
Alpamayo 1.5,1,1000,"schema_version, clip_ids, pred_xyz, gt_xyz, checkpoint, num_traj_sets, num_traj_samples, description, ade, min_ade, max_ade","(1000, 1, 6, 64, 3)",float32,"(1000, 64, 3)",float32,absent; manifest/log provenance,absent; manifest/log provenance
EoS 4B,1,1000,"schema_version, clip_ids, pred_xyz, gt_xyz, checkpoint, num_traj_sets, num_traj_samples, description, ade, min_ade, max_ade","(1000, 1, 6, 64, 3)",float32,"(1000, 64, 3)",float32,absent; manifest/log provenance,absent; manifest/log provenance
EoS 2B,1,1000,"schema_version, clip_ids, pred_xyz, gt_xyz, checkpoint, num_traj_sets, num_traj_samples, description, ade, min_ade, max_ade","(1000, 1, 6, 64, 3)",float32,"(1000, 64, 3)",float32,absent; manifest/log provenance,absent; manifest/log provenance
CM 4B,1,1000,"schema_version, clip_ids, pred_xyz, gt_xyz, checkpoint, num_traj_sets, num_traj_samples, description, ade, min_ade, max_ade","(1000, 1, 6, 64, 3)",float32,"(1000, 64, 3)",float32,absent; manifest/log provenance,absent; manifest/log provenance
CM 2B,1,1000,"schema_version, clip_ids, pred_xyz, gt_xyz, checkpoint, num_traj_sets, num_traj_samples, description, ade, min_ade, max_ade","(1000, 1, 6, 64, 3)",float32,"(1000, 64, 3)",float32,absent; manifest/log provenance,absent; manifest/log provenance


In [9]:
kd_runs = [
    ("Alpamayo 1.5", 1, "teacher15_nav_fp32archive_h100c_eos_models--nvidia--Alpamayo-1.5-10B-A1-format_nfe1", True),
    ("Alpamayo 1.5", 2, "teacher15_nav_fp32archive_h100c_eos_models--nvidia--Alpamayo-1.5-10B-A1-format_nfe2", True),
    ("Alpamayo 1.5", 10, "teacher_eval/teachernav_k10", False),
    ("KD 4B", 1, "kd4b_navstripped_h100c_control_checkpoint-13752_nfe1", True),
    ("KD 4B", 2, "kd4b_navstripped_h100c_control_checkpoint-13752_nfe2", True),
    ("KD 4B", 10, "stitch_4b_nav4bspan2camallfc_m1-9-18-36_framecache1080p_checkpoint-13752_cam13_nav", False),
    ("KD 2B", 1, "kd2b_navstripped_h100c_control_checkpoint-13752_nfe1", True),
    ("KD 2B", 2, "kd2b_navstripped_h100c_control_checkpoint-13752_nfe2", True),
    ("KD 2B", 10, "stitch_2b_mixspan2bnavfc_m1-9-18-36_framecache1080p_checkpoint-13752_cam13_nav", False),
]
kd_summary_rows = []
kd_provenance_rows = []
kd_per_clip_metrics = {}

for model_name, nfe, archive_stem, nav_stripping_verified in kd_runs:
    path = TRAINING_ROOT / f"{archive_stem}.npz"
    clip_ids, pred_xy, gt_xyz, inspection = load_archive(path, nfe)
    if not np.array_equal(clip_ids, reference_ids):
        raise ValueError(f"{path}: KD clip IDs differ from the main comparison")
    if not np.array_equal(gt_xyz, reference_gt):
        raise ValueError(f"{path}: KD ground truth differs from the main comparison")
    metrics = compute_metrics(pred_xy, gt_xyz)
    averages = metrics[METRIC_COLUMNS].mean()
    metrics.insert(0, "clip_id", clip_ids)
    kd_per_clip_metrics[(model_name, nfe)] = metrics
    kd_summary_rows.append({"Model": model_name, "NFE": nfe, **averages.to_dict()})
    kd_provenance_rows.append({
        "Model": model_name,
        "NFE": nfe,
        "NPZ": str(path),
        "Clips": len(clip_ids),
        "Student checkpoint": "checkpoint-13752",
        "Expert": "Unmodified Alpamayo 1.5 teacher, 36 layers",
        "Layer mixer": "Trained 28->36" if model_name == "KD 2B" else "None",
        "Navigation distance stripping": "Explicitly enabled" if nav_stripping_verified else "Historical: not verified",
        "Budget provenance": "Explicit step override in eval log" if nfe in (1, 2) else "No step override in eval log; sampler default is 10",
        **inspection,
    })

kd_results = pd.DataFrame(kd_summary_rows, columns=["Model", "NFE", *METRIC_COLUMNS])
kd_provenance = pd.DataFrame(kd_provenance_rows)
print(
    "KD evaluations: distilled VLM + untouched teacher expert, checkpoint-13752. "
    "NFE=1/2 use explicitly stripped navigation; historical NFE=10 preprocessing is not verified. "
    "This reference table is independent of the main table's NFE filter. "
    "All 1,000 clip IDs and saved GT match the main comparison."
)
kd_comparison_table = (
    kd_results.style
    .format({metric: "{:.2f}" for metric in METRIC_COLUMNS})
    .hide(axis="index")
    .set_caption("KD references: XY metres; green = lower, red = higher; bold = column minimum")
    .set_properties(subset=METRIC_COLUMNS, **{"text-align": "right"})
    .background_gradient(cmap="RdYlGn_r", subset=METRIC_COLUMNS, axis=0)
    .highlight_min(subset=METRIC_COLUMNS, axis=0, props="font-weight: bold;")
)
display(kd_comparison_table)
# display(kd_provenance.style.hide(axis="index").set_caption("KD archive provenance"))

combined_results = pd.concat([results, kd_results], ignore_index=True)

KD evaluations: distilled VLM + untouched teacher expert, checkpoint-13752. NFE=1/2 use explicitly stripped navigation; historical NFE=10 preprocessing is not verified. This reference table is independent of the main table's NFE filter. All 1,000 clip IDs and saved GT match the main comparison.


Model,NFE,ADE,min_ADE,max_ADE,center
Alpamayo 1.5,1,1.71,1.31,2.21,1.62
Alpamayo 1.5,2,1.47,0.95,2.14,1.36
Alpamayo 1.5,10,1.70,0.74,3.30,1.39
KD 4B,1,2.13,1.76,2.49,2.09
KD 4B,2,1.77,1.17,2.45,1.64
KD 4B,10,2.04,0.90,3.71,1.65
KD 2B,1,2.15,1.84,2.49,2.11
KD 2B,2,1.99,1.33,2.78,1.87
KD 2B,10,2.37,1.03,4.28,1.93


In [3]:
fullval_runs = {
    ("Alpamayo 1.5", 1): TRAINING_ROOT / "teacher15_availableval23331_fixedt0_stripped_nfe1_21170.npz",
    # ("Alpamayo 1.5", 2): TRAINING_ROOT / "teacher15_availableval23331_fixedt0_stripped_nfe2_21283.npz",
    ("KD 4B", 1): TRAINING_ROOT / "kd4b_availableval23331_fixedt0_stripped_ep4_nfe1_21686.npz",
    ("KD 2B", 1): TRAINING_ROOT / "kd2b_availableval23331_fixedt0_stripped_ep4_nfe1_21687.npz",
    ("EoS 2B", 1): TRAINING_ROOT / "eos2b_availableval23331_fixedt0_stripped_ep2_nfe1_21333.npz",
    ("EoS 4B", 1): TRAINING_ROOT / "eos4b_availableval23331_fixedt0_stripped_ep2_nfe1_21332.npz",
    # ("EoS 2B", 2): TRAINING_ROOT / "eos2b_availableval23331_fixedt0_stripped_ep2_nfe2_21284.npz",
    # ("EoS 4B", 2): TRAINING_ROOT / "eos4b_availableval23331_fixedt0_stripped_ep2_nfe2_21285.npz",
    ("CM 2B", 1): TRAINING_ROOT / "cm2b_availableval23331_fixedt0_stripped_ep2_nfe1_21167.npz",
    ("CM 4B", 1): TRAINING_ROOT / "cm4b_availableval23331_fixedt0_stripped_ep2_nfe1_21166.npz",
    # ("CM 2B", 2): TRAINING_ROOT / "cm2b_availableval23331_fixedt0_stripped_ep2_nfe2_21408.npz",
    # ("CM 4B", 2): TRAINING_ROOT / "cm4b_availableval23331_fixedt0_stripped_ep2_nfe2_21409.npz",
    
}
fullval_rows = []
fullval_per_clip_metrics = {}
fullval_reference_ids = None
fullval_reference_gt = None

for (fullval_model, fullval_expected_nfe), fullval_path in fullval_runs.items():
    with np.load(fullval_path, allow_pickle=False) as fullval_archive:
        fullval_ids = fullval_archive["clip_ids"].astype(str)
        fullval_pred = fullval_archive["pred_xyz"]
        fullval_gt = fullval_archive["gt_xyz"]
        if fullval_ids.shape != (23331,) or len(np.unique(fullval_ids)) != 23331:
            raise ValueError(f"{fullval_path}: expected 23,331 unique clip IDs")
        if fullval_pred.shape != (23331, 1, 6, 64, 3) or fullval_gt.shape != (23331, 64, 3):
            raise ValueError(f"{fullval_path}: unexpected prediction or ground-truth shape")
        if not np.isfinite(fullval_pred).all() or not np.isfinite(fullval_gt).all():
            raise ValueError(f"{fullval_path}: non-finite predictions or ground truth")
        if "cameras" in fullval_archive and not np.array_equal(fullval_archive["cameras"], [1, 3]):
            raise ValueError(f"{fullval_path}: expected cameras [1, 3]")
        if "inference_step" in fullval_archive:
            fullval_nfe = fullval_archive["inference_step"]
            if fullval_nfe.size != 1 or fullval_nfe.item() != fullval_expected_nfe:
                raise ValueError(f"{fullval_path}: expected NFE={fullval_expected_nfe}")

    fullval_order = np.argsort(fullval_ids)
    fullval_ids = fullval_ids[fullval_order]
    fullval_gt = fullval_gt[fullval_order]
    if fullval_reference_ids is None:
        fullval_reference_ids = fullval_ids.copy()
        fullval_reference_gt = fullval_gt.copy()
    elif not np.array_equal(fullval_ids, fullval_reference_ids) or not np.array_equal(fullval_gt, fullval_reference_gt):
        raise ValueError(f"{fullval_path}: clip IDs or saved ground truth differ across full-validation runs")

    fullval_metrics = compute_metrics(fullval_pred[fullval_order, 0, ..., :2], fullval_gt)
    fullval_metrics.index = pd.Index(fullval_ids, name="clip_id")
    fullval_per_clip_metrics[(fullval_model, fullval_expected_nfe)] = fullval_metrics
    fullval_rows.append({
        "Model": fullval_model,
        "NFE": fullval_expected_nfe,
        "Clips": len(fullval_ids),
        **fullval_metrics[METRIC_COLUMNS].mean().to_dict(),
    })
    del fullval_pred

fullval_results = pd.DataFrame(fullval_rows, columns=["Model", "NFE", "Clips", *METRIC_COLUMNS])
print(f"Verified 23,331 matching clips and identical saved GT across all {len(fullval_runs)} full-validation runs.")
print("Two cameras [1, 3]; t0=5.1 s; distance-free navigation; 427 missing-index clips excluded.")
print("Teacher and cotrained EoS have NFE=1/2 results; CM NFE=1 uses separate distilled checkpoints.")
fullval_comparison_table = (
    fullval_results.style
    .format({"Clips": "{:,}", **{metric: "{:.4f}" for metric in METRIC_COLUMNS}})
    .hide(axis="index")
    .set_caption("Full validation, NFE=1/2: XY metres; green = lower, red = higher; bold = column minimum")
    .set_properties(subset=METRIC_COLUMNS, **{"text-align": "right"})
    .background_gradient(cmap="RdYlGn_r", subset=METRIC_COLUMNS, axis=0)
    .highlight_min(subset=METRIC_COLUMNS, axis=0, props="font-weight: bold;")
)
display(fullval_comparison_table)

Verified 23,331 matching clips and identical saved GT across all 7 full-validation runs.
Two cameras [1, 3]; t0=5.1 s; distance-free navigation; 427 missing-index clips excluded.
Teacher and cotrained EoS have NFE=1/2 results; CM NFE=1 uses separate distilled checkpoints.


Model,NFE,Clips,ADE,min_ADE,max_ADE,center
Alpamayo 1.5,1,"23,331",1.6450,1.2327,2.1732,1.5572
KD 4B,1,"23,331",2.0121,1.6518,2.4048,1.9827
KD 2B,1,"23,331",2.0620,1.7551,2.4195,2.0305
EoS 2B,1,"23,331",1.7947,1.4814,2.1838,1.7361
EoS 4B,1,"23,331",1.5824,1.3600,1.8649,1.5425
CM 2B,1,"23,331",1.7116,1.1352,2.3815,1.6247
CM 4B,1,"23,331",1.5142,1.0368,2.0587,1.4480


In [ ]:
# fullval_results.to_csv("/home/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/notebooks/fullval_results.csv", index=False, float_format="%.8f")

In [6]:
SELECTED_MODELS = ["Alpamayo 1.5", "KD 4B", "EoS 4B", "CM 4B"]
SELECTED_NFE = [1]

filtered_fullval_results = fullval_results.loc[
    fullval_results["Model"].isin(SELECTED_MODELS) & fullval_results["NFE"].isin(SELECTED_NFE)
].copy()

filtered_table = (
    filtered_fullval_results.style
    .use(fullval_comparison_table.export())
    .format({metric: "{:.3f}" for metric in METRIC_COLUMNS})
    .hide(axis="index")
    .set_caption("Filtered trajectory metrics: XY metres; green = lower, red = higher; bold = column minimum")
)
display(filtered_table)

Model,NFE,Clips,ADE,min_ADE,max_ADE,center
Alpamayo 1.5,1,23331,1.645,1.233,2.173,1.557
KD 4B,1,23331,2.012,1.652,2.405,1.983
EoS 4B,1,23331,1.582,1.360,1.865,1.543
CM 4B,1,23331,1.514,1.037,2.059,1.448


In [7]:
from matplotlib.colors import LinearSegmentedColormap

FULLVAL_CATEGORY_METRIC = "min_ADE"
FULLVAL_SCENARIOS = Path(
    "/temp/achahe/physical_ai_av/lcdrive_physicalai_av_manifests/"
    "lcdrive_val_primary_scenario_for_table2.csv"
)
if FULLVAL_CATEGORY_METRIC not in METRIC_COLUMNS:
    raise ValueError(f"Choose a metric from {METRIC_COLUMNS}")

fullval_labels = pd.read_csv(
    FULLVAL_SCENARIOS, usecols=["clip_uuid", "scenario_category"], dtype=str
)
if fullval_labels["clip_uuid"].isna().any() or fullval_labels["clip_uuid"].duplicated().any():
    raise ValueError("Scenario labels must have one unique clip UUID per row")
fullval_categories = fullval_labels.set_index("clip_uuid")["scenario_category"].reindex(fullval_reference_ids)
if fullval_categories.isna().any() or fullval_categories.str.strip().eq("").any():
    raise ValueError("Missing primary scenario labels for full-validation clips")

fullval_category_scores = {}
for category_model, category_nfe in fullval_runs:
    category_metrics = fullval_per_clip_metrics[(category_model, category_nfe)]
    if not np.array_equal(category_metrics.index.to_numpy(), fullval_reference_ids):
        raise ValueError(f"{category_model}, NFE={category_nfe}: clip IDs differ from the full-validation comparison")
    fullval_category_scores[f"{category_model} (NFE={category_nfe})"] = category_metrics[FULLVAL_CATEGORY_METRIC].to_numpy()

fullval_category_frame = pd.DataFrame(
    fullval_category_scores, index=pd.Index(fullval_reference_ids, name="clip_id")
)
fullval_category_frame["category"] = fullval_categories.to_numpy()
fullval_model_columns = list(fullval_category_scores)
fullval_category_groups = fullval_category_frame.groupby("category")
fullval_category_table = fullval_category_groups[fullval_model_columns].mean()
fullval_category_table.insert(0, "n", fullval_category_groups.size())
fullval_category_table = fullval_category_table.sort_values("n", ascending=False)
if fullval_category_table["n"].sum() != 23331:
    raise ValueError("Category counts must sum to 23,331 clips")
fullval_category_overall = fullval_category_frame[fullval_model_columns].mean().to_frame().T
fullval_category_overall.insert(0, "n", len(fullval_category_frame))
fullval_category_overall.index = pd.Index(["ALL"], name="category")
np.testing.assert_allclose(
    fullval_category_overall.loc["ALL", fullval_model_columns].to_numpy(dtype=float),
    fullval_results.set_index(["Model", "NFE"]).reindex(list(fullval_runs))[FULLVAL_CATEGORY_METRIC].to_numpy(),
)
fullval_category_table = pd.concat([fullval_category_table, fullval_category_overall])

fullval_category_cmap = LinearSegmentedColormap.from_list(
    "fullval_categories", ["#f2f7fb", "#c8dbeb", "#7fa9cd", "#3d6f9e", "#1b3d5c"]
)
fullval_category_styled = (
    fullval_category_table.style
    .background_gradient(cmap=fullval_category_cmap, subset=fullval_model_columns, axis=1, text_color_threshold=0.45)
    .format({"n": "{:,.0f}", **{model: "{:.3f}" for model in fullval_model_columns}})
    .set_caption(
        f"Full validation: {FULLVAL_CATEGORY_METRIC}, XY metres, NFE shown per column; "
        "lower is better; colour is row-relative (per category)"
    )
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "0.95em"), ("padding-bottom", "8px"), ("color", "#444")]},
        {"selector": "th", "props": [("font-weight", "600"), ("text-align", "right")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
        {"selector": "td", "props": [("text-align", "right"), ("padding", "4px 10px")]},
    ])
)
print(f"{len(fullval_category_frame):,} matched clips; all primary categories present; ALL matches the full-validation table.")
print(fullval_category_table.round(3).to_string())
display(fullval_category_styled)

23,331 matched clips; all primary categories present; ALL matches the full-validation table.
                                    n  Alpamayo 1.5 (NFE=1)  KD 4B (NFE=1)  KD 2B (NFE=1)  EoS 2B (NFE=1)  EoS 4B (NFE=1)  CM 2B (NFE=1)  CM 4B (NFE=1)
category                                                                                                                                               
General Training/Validation      7960                 1.015          1.593          1.622           1.210           1.147          0.895          0.850
Lane Keeping Curve               1363                 1.478          1.971          2.222           1.843           1.659          1.509          1.374
Nudge Static Obstacle Maneuver   1313                 1.300          1.488          1.544           1.465           1.413          1.114          1.066
Lead Vehicle Following           1309                 1.205          1.700          1.803           1.631           1.431          1.264          1

,n,Alpamayo 1.5 (NFE=1),KD 4B (NFE=1),KD 2B (NFE=1),EoS 2B (NFE=1),EoS 4B (NFE=1),CM 2B (NFE=1),CM 4B (NFE=1)
category,,,,,,,,
General Training/Validation,"7,960",1.015,1.593,1.622,1.210,1.147,0.895,0.850
Lane Keeping Curve,"1,363",1.478,1.971,2.222,1.843,1.659,1.509,1.374
Nudge Static Obstacle Maneuver,"1,313",1.300,1.488,1.544,1.465,1.413,1.114,1.066
Lead Vehicle Following,"1,309",1.205,1.700,1.803,1.631,1.431,1.264,1.074
Nudge Maneuver,"1,243",1.283,1.425,1.517,1.436,1.337,1.096,1.007
Speed Control,"1,237",1.581,1.889,2.156,1.810,1.648,1.385,1.266
Lane Keeping,"1,221",1.242,1.516,1.728,1.620,1.417,1.271,1.094
Stop for Vehicle,"1,122",0.768,1.002,1.232,1.057,0.875,0.771,0.656
Vulnerable Road Users (VRU),"1,100",1.074,1.327,1.457,1.438,1.321,1.060,0.974


In [ ]:
# fullval_category_table.to_csv("/home/achahe/alpamayo-recipes/recipes/alpamayo1_5_distill/notebooks/fullval_category_table.csv", float_format="%.8f")